# Khám phá và Xử lý Dữ liệu (EDA) — Superstore Sales Dataset

Notebook này thực hiện đầy đủ quy trình EDA (25% điểm):
1. Đọc và kiểm tra chất lượng dữ liệu (`Sample - Superstore.csv`).
2. Làm sạch dữ liệu thiếu, dữ liệu trùng lặp, xử lý ngoại lệ và định dạng lại cột thời gian/doanh thu.
3. Thống kê tổng hợp bằng `groupby()` và `pd.pivot_table()`.
4. Trực quan hóa khám phá ban đầu bằng Plotly.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.append(str(Path('..').resolve()))
from data.loader import audit_raw_data, clean_superstore_data, load_raw_data
from data.analytics import (
    compute_rfm_segments,
    forecast_monthly_sales,
    pivot_region_category,
    pivot_year_segment,
    run_market_basket_apriori,
    summarize_by_category,
    summarize_by_region,
    summarize_by_time,
)

## 1. Đọc dữ liệu và Kiểm tra chất lượng (Data Audit)

In [ ]:
raw_df = load_raw_data('../Sample - Superstore.csv')
print('Kích thước dữ liệu gốc:', raw_df.shape)
raw_df.head()

In [ ]:
audit = audit_raw_data(raw_df)
for k, v in audit.items():
    print(f'{k}: {v}')

## 2. Làm sạch dữ liệu, Định dạng cột và Xử lý ngoại lệ

In [ ]:
clean_df = clean_superstore_data(raw_df, cap_outliers=False)
print('Kích thước sau làm sạch:', clean_df.shape)
clean_df[['Order Date', 'Ship Date', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']].info()

In [ ]:
clean_df[['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping_Days']].describe().round(2)

### Lọc dữ liệu theo điều kiện

In [ ]:
loss_df = clean_df[clean_df['Profit'] < 0]
high_disc_loss = clean_df[(clean_df['Discount'] > 0.2) & (clean_df['Profit'] < 0)]
print(f'Số dòng giao dịch thua lỗ: {len(loss_df)}')
print(f'Số dòng chiết khấu > 20% gây lỗ: {len(high_disc_loss)}')

## 3. Thống kê & Tổng hợp bằng GroupBy và Pivot Table

In [ ]:
print('--- GroupBy theo Năm ---')
display(summarize_by_time(clean_df, freq='Y'))

print('--- GroupBy theo Khu vực ---')
display(summarize_by_region(clean_df))

print('--- GroupBy theo Danh mục ---')
display(summarize_by_category(clean_df))

In [ ]:
print('--- Pivot Table: Khu vực x Danh mục (Doanh thu) ---')
display(pivot_region_category(clean_df).round(2))

print('--- Pivot Table: Năm x Phân khúc (Doanh thu & Số đơn hàng) ---')
display(pivot_year_segment(clean_df).round(2))

## 4. Trực quan hóa dữ liệu bắt buộc (So sánh, Phân phối, Xu hướng, Tổng hợp)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(clean_df['Sales'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('Biểu đồ Phân phối (Histogram) - Sales')
sns.boxplot(data=clean_df, x='Category', y='Profit', ax=axes[1])
axes[1].set_title('Biểu đồ Phân phối & Ngoại lệ (Boxplot) - Profit theo Category')
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ so sánh (Bar) và biểu đồ tổng hợp (Stacked/Grouped Bar)
reg_df = summarize_by_region(clean_df)
px.bar(reg_df, x='Region', y='Total_Sales', color='Region', title='Biểu đồ So sánh Doanh thu theo Khu vực').show()

cat_seg = clean_df.groupby(['Segment', 'Category'], as_index=False)['Sales'].sum()
px.bar(cat_seg, x='Segment', y='Sales', color='Category', barmode='stack', title='Biểu đồ Tổng hợp (Stacked Bar) Segment x Category').show()

In [ ]:
monthly_df = summarize_by_time(clean_df, freq='M')
fig = px.line(monthly_df, x='Period', y=['Total_Sales', 'Total_Profit'], title='Biểu đồ Xu hướng Doanh thu & Lợi nhuận theo Tháng')
fig.show()

In [ ]:
fig_disc = px.scatter(
    clean_df,
    x='Discount',
    y='Profit',
    color='Category',
    title='Tác động của Chiết khấu đến Lợi nhuận'
)
fig_disc.add_hline(y=0, line_dash='dash', line_color='red')
fig_disc.show()